# Neuronales Netz - Aktivierungsfunktion, Lernrate, Batch-Size, Epochenanzahl

**systematische Einschränkung der Hyperparameterbreiche**

**ki-308** California House Prising

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.linear_model import LinearRegression

from utils.data import load_and_clean_data, get_train_test_split
from utils.evaluation import evaluate_predictions, add_result
from utils.plotting import plot_features_vs_target, plot_predicted_vs_actual, plot_residuals, save_fig

plt.rcParams['figure.dpi'] = 100
%matplotlib inline

print(f"TensorFlow Version: {tf.__version__}")
print(f"Numpy Version: {np.__version__}")
print(f"Pandas Version: {pd.__version__}")
print(f"Matplotlib Version: {plt.matplotlib.__version__}")
print(f"python Version: {sys.version}")

In [ ]:
def build_model(hidden_layers, activation='relu', learning_rate=0.001):
    """Erstelle ein Sequential-Modell mit gegebener Architektur."""
    model = keras.Sequential()
    model.add(layers.Input(shape=(X_train.shape[1],)))
    
    for units in hidden_layers:
        model.add(layers.Dense(units, activation=activation))
    
    model.add(layers.Dense(1))  # Regression Output
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae'],
    )
    return model


def train_and_evaluate(model, name, x_train, y_train, x_val, y_val, x_test, y_test,
                       epochs=100, batch_size=32, verbose=0):

    history = model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        verbose=verbose,
    )

    y_train_pred = model.predict(x_train, verbose=0).flatten()
    y_test_pred = model.predict(x_test, verbose=0).flatten()

    result = evaluate_predictions(y_train, y_train_pred, y_test, y_test_pred, name)
    add_result(result)

    return history, result

## Daten laden und transformieren

Da wir sehr verschiedene Intervalle haben sollte eine Skalierung sehr sinnvoll sein, um ansonsten z.B. Sigmoid nicht gut funktionieren kann.

In [ ]:
df = load_and_clean_data()
X_train, X_test, y_train, y_test, feature_names = get_train_test_split(df)

# Validierungssplit aus Trainingsdaten
val_split = int(0.8 * len(X_train))
X_val, y_val = X_train[val_split:], y_train[val_split:]
X_train, y_train = X_train[:val_split], y_train[:val_split]

print(f"Training:   {X_train.shape}")
print(f"Validation: {X_val.shape}")
print(f"Test:       {X_test.shape}")
print(f"Features:   {feature_names}")

In [ ]:
X_train_standard, X_test_standard, y_train_standard, y_test_standard,scaler, feature_names = X_train, X_test, y_train, y_test, scaler, feature_names = get_train_test_split(df, scaler='standard')

val_split = int(0.8 * len(X_train_standard))
X_val_standard, y_val_standard = X_train_standard[val_split:], y_train_standard[val_split:]
X_train_standard, y_train_standard = X_train_standard[:val_split], y_train_standard[:val_split]

print(f"Training:   {X_train_standard.shape}")
print(f"Validation: {X_val_standard.shape}")
print(f"Test:       {X_test_standard.shape}")
print(f"Features:   {feature_names}")

In [ ]:
X_train_min0_max1, X_test_min0_max1, y_train_min0_max1, y_test_min0_max1, scaler, feature_names = get_train_test_split(df, scaler='minmax')

val_split = int(0.8 * len(X_train_min0_max1))
X_val_min0_max1, y_val_min0_max1 = X_train_min0_max1[val_split:], y_train_min0_max1[val_split:]
X_train_min0_max1, y_train_min0_max1 = X_train_min0_max1[:val_split], y_train_min0_max1[:val_split]

print(f"Training:   {X_train_min0_max1.shape}")
print(f"Validation: {X_val_min0_max1.shape}")
print(f"Test:       {X_test_min0_max1.shape}")
print(f"Features:   {feature_names}")



Beginn mit ReLu: (https://www.datacamp.com/de/tutorial/introduction-to-activation-functions-in-neural-networks?dc_referrer=https%3A%2F%2Fwww.google.com%2F)

- rechnerisch kostengünstig auch bei Anwendung mehrerer Schichten
- gängiste Aktivierungsfunktion

Fixparameter:
- Loss: MSE
- Hidden Layers: 2 mit 64 und 32 units (Datensatz zu klein für tieferes Netz und zu geringe Features für breiteres Netz; abnehmende Größen beugen Overfitting vor [Copilot Promt: welche fixparameter wie layer-, unitanzahl etc. sollte man hier wählen?])
- Metric: mae
- learning_rate: 0.001
- Epochen: 100 (von Florian übernommen)
- Batch-Size: 32 (von Florian übernommen)

In [ ]:
model = keras.Sequential()
model.add(layers.Input(shape=(X_train_min0_max1.shape[1],)))
model.add(layers.Dense(64, activation="relu"))
model.add(layers.Dense(32, activation="relu"))
model.add(layers.Dense(1))  # Regression Output
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae'],
)
train_and_evaluate(model, "NN_min0_max1_relu", X_train_min0_max1, y_train_min0_max1, X_val_min0_max1, y_val_min0_max1, X_test_min0_max1, y_test_min0_max1, epochs=100, batch_size=32, verbose=0)


## Beste Kombination aus Aktivierungsfunktion und Skalierung finden

Diese Zeile muss ausgeführt werden, damit später x_train etc. überhaupt definiert ist.

In [ ]:
activations = ['relu', 'tanh', 'sigmoid', 'elu', 'selu', 'leaky_relu']
dataset = [("normal", X_train, y_train, X_val, y_val, X_test, y_test),
           ("standard", X_train_standard, y_train_standard, X_val_standard, y_val_standard, X_test_standard, y_test_standard),
           ("minmax", X_train_min0_max1, y_train_min0_max1, X_val_min0_max1, y_val_min0_max1, X_test_min0_max1, y_test_min0_max1)]

for activation in activations:
    for scale_name, x_train, y_train, x_val, y_val, x_test, y_test in dataset:
        model_name = f"NN_{activation}_{scale_name}"
        model = keras.Sequential()
        model.add(layers.Input(shape=(x_train.shape[1],)))
        model.add(layers.Dense(64, activation=activation))
        model.add(layers.Dense(32, activation=activation))
        model.add(layers.Dense(1))  # Regression Output
        model.compile(
            optimizer=keras.optimizers.Adam(learning_rate=0.001),
            loss='mse',
            metrics=['mae'],
        )
        train_and_evaluate(model, model_name, x_train, y_train, x_val, y_val, x_test, y_test, epochs=100, batch_size=32, verbose=0)        



Wenn man sich nur den Testscore anschaut gewinnt der Tanh + Standardscaler, was logisch ist, da der Tanh daten um die Null herum erwartet und etwas glatter ist und daher nicht so zu overfitting neigt (siehe Relu 6% Abweichung Training und Test). Daher als beste Kombination angesehen

## Lernrate, Batch-size und Epochenanzahl

Weitere Fixparameter sind nun: Tanh und Standardscaler

Vorgehen wie in diesem Artikel beschrieben: https://www.kdnuggets.com/tuning-hyperparameters-in-neural-networks?utm_source=copilot.com

Mit der Lernrate anfangen für bessere Konvergenz -> erstes händisches Gefühl für die richtige Größenordnung

In [ ]:
learning_rate = [0.000001, 0.00001, 0.0001, 0.001, 0.01, 0.1, 0.5, 1.0, 10.0]

for learning_rate in learning_rate:

    model_name = f"NN_LearningRate_{learning_rate}"
    model = keras.Sequential()
    model.add(layers.Input(shape=(x_train.shape[1],)))
    model.add(layers.Dense(64, activation="tanh"))
    model.add(layers.Dense(32, activation="tanh"))
    model.add(layers.Dense(1))  # Regression Output
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae'],
    )
    train_and_evaluate(model, model_name, x_train, y_train, x_val, y_val, x_test, y_test, epochs=100, batch_size=32, verbose=0)    

Aus obigen Ergebnissen in folgendem Bereich genauer suchen: zwischen 0.001 - 0.01

In [ ]:
learning_rate = np.linspace(0.001, 0.01, 50)

for learning_rate in learning_rate:

    model_name = f"NN_LearningRate_{learning_rate}"
    model = keras.Sequential()
    model.add(layers.Input(shape=(x_train.shape[1],)))
    model.add(layers.Dense(64, activation="tanh"))
    model.add(layers.Dense(32, activation="tanh"))
    model.add(layers.Dense(1))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae'],
    )
    train_and_evaluate(model, model_name, x_train, y_train, x_val, y_val, x_test, y_test, epochs=100, batch_size=32, verbose=0)  

Die Auswertung über die Lernraten zeigt kein globales Optimum sondern nur lokale Minima, die für genau dieses Modell: Aktivierungsfkt., Batch-Size, etc. gelten. Daher sollte die Lernrate im Finalen Modell erneut als Hyperparameter getestet werden. Das momentane Optimum liegt bei 0,0054

Test, ob 0,0054 ebenso gut ist wie 0,005408163265306123

In [ ]:
learning_rate = (0.0054, 0.005408163265306123)

for learning_rate in learning_rate:

    model_name = f"NN_LearningRate_{learning_rate}"
    model = keras.Sequential()
    model.add(layers.Input(shape=(x_train.shape[1],)))
    model.add(layers.Dense(64, activation="tanh"))
    model.add(layers.Dense(32, activation="tanh"))
    model.add(layers.Dense(1))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae'],
    )
    train_and_evaluate(model, model_name, x_train, y_train, x_val, y_val, x_test, y_test, epochs=100, batch_size=32, verbose=0)  

Und was passiert bei einfach nur erneuten Durchläufen des selben Codes?

In [ ]:
learning_rate = (0.0054, 0.005408163265306123)

for learning_rate in learning_rate:

    model_name = f"NN_LearningRate_{learning_rate}"
    model = keras.Sequential()
    model.add(layers.Input(shape=(x_train.shape[1],)))
    model.add(layers.Dense(64, activation="tanh"))
    model.add(layers.Dense(32, activation="tanh"))
    model.add(layers.Dense(1))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae'],
    )
    train_and_evaluate(model, model_name, x_train, y_train, x_val, y_val, x_test, y_test, epochs=100, batch_size=32, verbose=0)  

In [ ]:
learning_rate = (0.0054, 0.005408163265306123)

for learning_rate in learning_rate:

    model_name = f"NN_LearningRate_{learning_rate}"
    model = keras.Sequential()
    model.add(layers.Input(shape=(x_train.shape[1],)))
    model.add(layers.Dense(64, activation="tanh"))
    model.add(layers.Dense(32, activation="tanh"))
    model.add(layers.Dense(1))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae'],
    )
    train_and_evaluate(model, model_name, x_train, y_train, x_val, y_val, x_test, y_test, epochs=100, batch_size=32, verbose=0)  

In [ ]:
learning_rate = (0.0054, 0.005408163265306123)

for learning_rate in learning_rate:

    model_name = f"NN_LearningRate_{learning_rate}"
    model = keras.Sequential()
    model.add(layers.Input(shape=(x_train.shape[1],)))
    model.add(layers.Dense(64, activation="tanh"))
    model.add(layers.Dense(32, activation="tanh"))
    model.add(layers.Dense(1))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='mse',
        metrics=['mae'],
    )
    train_and_evaluate(model, model_name, x_train, y_train, x_val, y_val, x_test, y_test, epochs=100, batch_size=32, verbose=0)  

Resultate des mehrmaligen Durchlaufes: Abweichung von bis zu 3 % beobachtet. Daher unerhablich welche Verbesserungen bis jetzt durch Hyperparametrisierung erreicht worden sind.
-> Initialgewichte Keras
-> Adam nutzt zufällige Initialisierung und unterschiedliche Ausführungsreihenfolge auf CPU/GPU
-> Reproduzierbarkeit von Floating-Points

Idee: Statistische Untersuchung von Mittelwert und Abweichung bei erneuter Ausführung des selben Modells:

In [ ]:
data = []

for i in range(1,101):

    model_name = f"NN_model_{i}"
    model = keras.Sequential()
    model.add(layers.Input(shape=(x_train.shape[1],)))
    model.add(layers.Dense(64, activation="tanh"))
    model.add(layers.Dense(32, activation="tanh"))
    model.add(layers.Dense(1))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0054),
        loss='mse',
        metrics=['mae'],
    )
    history, result = train_and_evaluate(model, model_name, x_train, y_train, x_val, y_val, x_test, y_test, epochs=100, batch_size=32, verbose=0)  
    data.append(result['R² Test'])

bin_heights, bin_borders = np.histogram(data, bins=30, density=True)
bin_centers = (bin_borders[:-1] + bin_borders[1:]) / 2

def gaussian(x, A, mu, sigma):
    return A * np.exp(- ((x - mu)**2 / (2 * sigma**2))) 

mean_guess = np.mean(data)
sigma_guess = np.std(data)
max_guess = np.max(bin_heights)

popt, pcov = curve_fit(gaussian, bin_centers, bin_heights, p0=[max_guess, mean_guess, sigma_guess])
fitted_A, fitted_mu, fitted_sigma = popt
fit_variance = fitted_sigma ** 2

print(f"Mittelwert (mu): {fitted_mu:.4f}")
print(f"Standardabweichung (sigma): {fitted_sigma:.4f}")
print(f"Varianz: {fit_variance:.4f}")

plt.hist(data, bins=30, alpha=0.6, color='g', label='R² Test Scores')
x_fit = np.linspace(min(bin_centers), max(bin_centers), 100)
plt.plot(x_fit, gaussian(x_fit, *popt), 'r-', label='Gaussian Fit')
plt.legend()
plt.xlabel('R² Test Score')
plt.ylabel('Häufigkeit')
plt.title('Verteilung der R² Test Scores mit Gaussian Fit')
plt.show()

In [ ]:
#from scipy.optimize import curve_fit

#bin_heights, bin_borders = np.histogram(data, bins=10, density=True)
#bin_centers = (bin_borders[:-1] + bin_borders[1:]) / 2

#def gaussian(x, A, mu, sigma):
    #return A * np.exp(- ((x - mu)**2 / (2 * sigma**2))) 

#mean_guess = np.mean(data)
#sigma_guess = np.std(data)
#max_guess = np.max(bin_heights)

#popt, pcov = curve_fit(gaussian, bin_centers, bin_heights, p0=[max_guess, mean_guess, sigma_guess])
#fitted_A, fitted_mu, fitted_sigma = popt
#fit_variance = fitted_sigma ** 2

#print(f"Mittelwert (mu): {fitted_mu:.4f}")
#print(f"Standardabweichung (sigma): {fitted_sigma:.4f}")
#print(f"Varianz: {fit_variance:.4f}")

plt.hist(data, bins=50, alpha=0.6, color='g')
#x_fit = np.linspace(min(bin_centers), max(bin_centers), 100)
#plt.plot(x_fit, gaussian(x_fit, *popt), 'r-', label='Gaussian Fit')
#plt.legend()
#plt.xlabel('R² Test Score')
plt.ylabel('Häufigkeit')
plt.title('Verteilung der R² Test Scores mit Gaussian Fit')
plt.show()

Mittelwert und Standardabweichung bestimmen

In [ ]:
sum = 0
for i in range(100):
    sum += data[i]

mittelwert = sum / 100

print(f"Mittelwert: {mittelwert:.4f}")

sum_quadrate = 0
for j in range(100):
    sum_quadrate += (data[j] - mittelwert) ** 2

varianz = (sum_quadrate / 100)
print(f"Varianz: {varianz:.4f}")
standardabweichung = varianz ** 0.5
print(f"Standardabweichung: {standardabweichung:.4f}")

## Batch-Size

In [ ]:
batch_size = [4, 8, 16, 32, 64, 128, 256, 512]

for batch_size in batch_size:

    model_name = f"NN_BatchSize_{batch_size}"
    model = keras.Sequential()
    model.add(layers.Input(shape=(x_train.shape[1],)))
    model.add(layers.Dense(64, activation="tanh"))
    model.add(layers.Dense(32, activation="tanh"))
    model.add(layers.Dense(1))  # Regression Output
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0054),
        loss='mse',
        metrics=['mae'],
    )
    train_and_evaluate(model, model_name, x_train, y_train, x_val, y_val, x_test, y_test, epochs=100, batch_size=batch_size, verbose=0) 

### Epochenanzahl

In [ ]:
epochs = [10, 50, 100, 200, 500, 1000, 10000]

for epochs in epochs:

    model_name = f"NN_Epochs_{epochs}"
    model = keras.Sequential()
    model.add(layers.Input(shape=(x_train.shape[1],)))
    model.add(layers.Dense(64, activation="tanh"))
    model.add(layers.Dense(32, activation="tanh"))
    model.add(layers.Dense(1))  # Regression Output
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0054),
        loss='mse',
        metrics=['mae'],
    )
    train_and_evaluate(model, model_name, x_train, y_train, x_val, y_val, x_test, y_test, epochs=epochs, batch_size=32, verbose=0) 

In [ ]:
# Modell definieren
model = keras.Sequential([
    layers.Input(shape=(x_train.shape[1],)),
    layers.Dense(64, activation="tanh"),
    layers.Dense(32, activation="tanh"),
    layers.Dense(1)
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0054),
    loss='mse',
    metrics=['mae'],
)

# Listen zum Speichern der Test-Scores
test_mae_per_epoch = []
test_mse_per_epoch = []

# Training über 500 Epochen, aber jede Epoche einzeln
for epoch in range(1, 501):

    model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=1,
        batch_size=32,
        verbose=0,
        shuffle=False
    )

    # Testdaten evaluieren
    mse, mae = model.evaluate(x_test, y_test, verbose=0)
    test_mae_per_epoch.append(mae)
    test_mse_per_epoch.append(mse)

# Plot
plt.figure(figsize=(10, 5))
plt.plot(test_mae_per_epoch, label="Test MAE")
plt.xlabel("Epoche")
plt.ylabel("MAE")
plt.title("Test-MAE pro Epoche (1–500)")
plt.grid(True)
plt.legend()
plt.show()


Wir sind bei circa 500 Epochen bei einem Minimum angekommen, daher sollten mehrere Epochen untersucht werden. Ebenfalls wird sich der Verlauf des Test-scores angeschaut.

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(x_train.shape[1],)),
    layers.Dense(64, activation="tanh"),
    layers.Dense(32, activation="tanh"),
    layers.Dense(1)
])

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0054),
    loss='mse',
    metrics=['mae'],
)

test_mae_per_epoch = []
test_R2_per_epoch = []

for epoch in range(1, 1001):

    model.fit(
        x_train, y_train,
        validation_data=(x_val, y_val),
        epochs=1,
        batch_size=32,
        verbose=0,
        shuffle=False
    )

    mse, mae = model.evaluate(x_test, y_test, verbose=0)
    y_test_predictions = model.predict(x_test, verbose=0).flatten()
    ss_res = np.sum((y_test - y_test_predictions) ** 2) 
    ss_tot = np.sum((y_test - np.mean(y_test)) ** 2) 
    r2_score = 1 - (ss_res / ss_tot)
    test_R2_per_epoch.append(r2_score)
    test_mae_per_epoch.append(mae)
    
plt.figure(figsize=(10, 5))
plt.plot(test_mae_per_epoch, label="Test MAE")
plt.plot(test_R2_per_epoch, label="Test R²")
plt.xlabel("Epoche")
plt.ylabel("Score")
plt.title("Test-MAE und Test-R² pro Epoche (1–1000)")
plt.grid(True)
plt.legend()
plt.show()

## Verbesserung der Hyperparameter mit Random Search

Random Search wird Grid Search vorgezogen (https://www.jmlr.org/papers/volume13/bergstra12a/bergstra12a.pdf)

In [ ]:
def sample_parameters():
    return {
        "learning_rate": np.random.uniform(0.001, 0.01),
        "batch_size": np.random.choice([16, 32, 64, 128]),
        "epochs": np.random.choice([100, 200, 300, 400, 500]),
        "activation": np.random.choice(['relu', 'tanh', 'sigmoid', 'elu', 'selu', 'leaky_relu'])
    }

In [ ]:
def build_model(hyperparameters, input_dim):
    model_name = f"NN_LR_{hyperparameters['learning_rate']}_BS_{hyperparameters['batch_size']}_Epochs_{hyperparameters['epochs']}_Akt_{hyperparameters['activation']}"
    model = keras.Sequential()
    model.add(layers.Input(shape=(input_dim,)))
    model.add(layers.Dense(64, activation=hyperparameters['activation']))
    model.add(layers.Dense(32, activation=hyperparameters['activation']))
    model.add(layers.Dense(1))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=hyperparameters['learning_rate']),
        loss='mse',
        metrics=['mae'],
    )
    #train_and_evaluate(model, model_name, X_train_standard, y_train_standard, X_val_standard, y_val_standard, x_test_standard, y_test_standard, epochs=hyperparameters['epochs'], batch_size=hyperparameters['batch_size'], verbose=0)  
    return model

In [ ]:
def random_search(X_train_standard, y_train_standard, X_val_standard, y_val_standard, x_test_standard, y_test_standard, n_iterations=10):
    results = []
    best_val_score = -float('inf')
    best_model_random_search = None
    best_hyperparameters = None

    input_dim = X_train_standard.shape[1]

    for i in range(n_iterations):
        hyperparameters = sample_parameters()
        print(f"Iteration {i+1}/{n_iterations} - Hyperparameter: {hyperparameters}")

        model = build_model(hyperparameters, input_dim)

        history, result = train_and_evaluate(
            model, 
            f"RandomSearch_{i+1}", 
            x_train=X_train_standard, 
            y_train=y_train_standard, 
            x_val=X_val_standard, 
            y_val=y_val_standard, 
            x_test=x_test_standard, 
            y_test=y_test_standard, 
            epochs=hyperparameters['epochs'], 
            batch_size=hyperparameters['batch_size'], 
            verbose=0
        )
        val_mae = history.history['val_mae'][-1]
        test_r2 = result['R² Test']

        print(f"Val MAE: {val_mae:.4f} - Test R²: {test_r2:.4f}")
        
        results.append({
            "val_mae": val_mae,
            "test_r2": test_r2,
            "hyperparameters": hyperparameters,
        })

        if test_r2 > best_val_score:
            best_val_score = test_r2
            best_model_random_search = model
            best_hyperparameters = hyperparameters
        
        
        print(f"Bestes Modell nach Iteration {i+1} mit Test score: {best_val_score:.4f} und Hyperparameter: {best_hyperparameters}")
        
    best_model_random_search.save(f"best_model_random_search_iterations_{i+1}.h5")
    print(f"\n bestes Modell gespeichert unter: best_model_random_search_iterations_{i+1}.h5")

    return sorted(results, key=lambda x: x["test_r2"], reverse=True)
            

In [ ]:
def r2_score(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2) 
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2) 
    return 1 - (ss_res / ss_tot)

In [ ]:
best_results = random_search(X_train_standard, y_train_standard, X_val_standard, y_val_standard, X_test_standard, y_test_standard, n_iterations=100) 
print("Beste Ergebnisse:")
for result in best_results:
    print(f"Val MAE: {result['val_mae']:.4f} - Test R²: {result['test_r2']:.4f} - Hyperparameter: {result['hyperparameters']}")

Das beste Modell noch einmal prüfen und Abspeichern testen/überprüfen

In [ ]:
model_load = keras.models.load_model("../results/nn_models/best_model_random_search_iterations_100.h5")

y_train_predictions = model_load.predict(X_train_standard, verbose=0).flatten()
y_test_predictions = model_load.predict(X_test_standard, verbose=0).flatten()

results = evaluate_predictions(y_train_standard, y_train_predictions, y_test_standard, y_test_predictions, "Bestes Modell Random Search")

## Learning-Rate-Schedule

*learning-rate wird während des Trainings automatisch angebracht*

KI-Promt: Verbesserungen an Hyperparameter liegen innerhalb Schwankungen des Trainings, welche Möglichkeiten gibt es das Projekt weiter zu verbessern? > Was ist LR-Schedule?

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

lr_scheduler = ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.5, 
    patience=5, 
    verbose=1
)

In [ ]:
model_1 = keras.Sequential([
    layers.Input(shape=(X_train_standard.shape[1],)),
    layers.Dense(64, activation="tanh"),
    layers.Dense(32, activation="tanh"),
    layers.Dense(1)
])
model_2 = keras.Sequential([
    layers.Input(shape=(X_train_standard.shape[1],)),
    layers.Dense(64, activation="tanh"),
    layers.Dense(32, activation="tanh"),
    layers.Dense(1)
])

model_1.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0054),
    loss='mse',
    metrics=['mae'],
)
model_2.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0054),
    loss='mse',
    metrics=['mae'],
)

test_mae_per_epoch_1 = []
test_mae_per_epoch_2 = []
test_R2_per_epoch_1 = []
test_R2_per_epoch_2 = []

for epoch in range(1, 501):

    model_1.fit(
        X_train_standard, y_train_standard,
        validation_data=(X_val_standard, y_val_standard),
        epochs=1,
        batch_size=32,
        verbose=0,
        shuffle=True
    )

    model_2.fit(
        X_train_standard, y_train_standard,
        validation_data=(X_val_standard, y_val_standard),
        epochs=1,
        batch_size=32,
        verbose=0,
        shuffle=True,
        callbacks=[lr_scheduler]
    )

    mse_1, mae_1 = model_1.evaluate(X_test_standard, y_test_standard, verbose=0)
    mse_2, mae_2 = model_2.evaluate(X_test_standard, y_test_standard, verbose=0)
    y_test_predictions_1 = model_1.predict(X_test_standard, verbose=0).flatten()
    y_test_predictions_2 = model_2.predict(X_test_standard, verbose=0).flatten()
    ss_res_1 = np.sum((y_test_standard - y_test_predictions_1) ** 2) 
    ss_res_2 = np.sum((y_test_standard - y_test_predictions_2) ** 2)
    ss_tot = np.sum((y_test_standard - np.mean(y_test_standard)) ** 2) 
    r2_score_1 = 1 - (ss_res_1 / ss_tot)
    r2_score_2 = 1 - (ss_res_2 / ss_tot)
    test_R2_per_epoch_1.append(r2_score_1)
    test_R2_per_epoch_2.append(r2_score_2)
    test_mae_per_epoch_1.append(mae_1)
    test_mae_per_epoch_2.append(mae_2)

    
plt.figure(figsize=(10, 5))
plt.plot(test_mae_per_epoch_1, label="Test MAE Model 1")
plt.plot(test_mae_per_epoch_2, label="Test MAE Model 2")
plt.plot(test_R2_per_epoch_1, label="Test R² Model 1")
plt.plot(test_R2_per_epoch_2, label="Test R² Model 2")
plt.xlabel("Epoche")
plt.ylabel("Score")
plt.title("Test-MAE und Test-R² pro Epoche (1–500)")
plt.grid(True)
plt.legend()
plt.show()

KI-Promt: Warum unterscheiden sich die Modelle nicht wesentlich?

Das Problem ist, dass immer nur über eine Epoche trainiert wird und daher die Lernrate nicht angepasst werden kann, daher folgende Codeverbesserung von KI:

In [ ]:
EPOCHS = 500
BLOCK = 10

for start in range(0, EPOCHS, BLOCK):
    end = min(start + BLOCK, EPOCHS)

    history_1 = model_1.fit(
        X_train_standard, y_train_standard,
        validation_data=(X_val_standard, y_val_standard),
        epochs=end,
        initial_epoch=start,
        batch_size=32,
        verbose=0
    )

    history_2 = model_2.fit(
        X_train_standard, y_train_standard,
        validation_data=(X_val_standard, y_val_standard),
        epochs=end,
        initial_epoch=start,
        batch_size=32,
        verbose=0,
        callbacks=[lr_scheduler]
    )

    # Jetzt kannst du pro Epoche loggen
    for epoch in range(start, end):
        mse1, mae1 = model_1.evaluate(X_test_standard, y_test_standard, verbose=0)
        mse2, mae2 = model_2.evaluate(X_test_standard, y_test_standard, verbose=0)

        y_pred1 = model_1.predict(X_test_standard, verbose=0).flatten()
        y_pred2 = model_2.predict(X_test_standard, verbose=0).flatten()

        r2_1 = 1 - np.sum((y_test_standard - y_pred1)**2) / ss_tot
        r2_2 = 1 - np.sum((y_test_standard - y_pred2)**2) / ss_tot

        test_mae_per_epoch_1.append(mae1)
        test_mae_per_epoch_2.append(mae2)
        test_R2_per_epoch_1.append(r2_1)
        test_R2_per_epoch_2.append(r2_2)


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(test_mae_per_epoch_1[:500], label="Test MAE kein LR-Schedule")
plt.plot(test_mae_per_epoch_2[:500], label="Test MAE mit LR-Schedule")
plt.plot(test_R2_per_epoch_1[:500], label="Test R² kein LR-Schedule")
plt.plot(test_R2_per_epoch_2[:500], label="Test R² mit LR-Schedule")
plt.xlabel("Epoche")
plt.ylabel("Score")
plt.title("Test-MAE und Test-R² pro Epoche (1–500)")
plt.grid(True)
plt.legend()
plt.show()

## Training auf die wichtigsten Features

In [ ]:
important_features = ["MedInc", "AveOccup", "Latitude", "Longitude"] # aus rf_feauture_importance.png
# In Indizes umwandeln für Numpy Arrays
important_indices = [feature_names.index(feature) for feature in important_features]
X_train_important = X_train_standard[:, important_indices]
y_train_important = y_train_standard[:]
X_test_important = X_test_standard[:, important_indices]
y_test_important = y_test_standard[:]
X_val_important = X_val_standard[:, important_indices]
y_val_important = y_val_standard[:]

for i in range(5):

    model_name = f"NN_important_features_iteration_{i}"
    model = keras.Sequential()
    model.add(layers.Input(shape=(X_train_important.shape[1],)))
    model.add(layers.Dense(64, activation="tanh"))
    model.add(layers.Dense(32, activation="tanh"))
    model.add(layers.Dense(1))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.005),
        loss='mse',
        metrics=['mae'],
    )
    train_and_evaluate(model, model_name, X_train_important, y_train_important, X_val_important, y_val_important, X_test_important, y_test_important, epochs=300, batch_size=32, verbose=0)  


Nur das Feature mit der stärksten Korrelation: MedInc

In [ ]:
important_features = ["MedInc"] # aus rf_feauture_importance.png
# In Indizes umwandeln für Numpy Arrays
important_indices = [feature_names.index(feature) for feature in important_features]
X_train_important = X_train_standard[:, important_indices]
y_train_important = y_train_standard[:]
X_test_important = X_test_standard[:, important_indices]
y_test_important = y_test_standard[:]
X_val_important = X_val_standard[:, important_indices]
y_val_important = y_val_standard[:]

results = []
best_test_score = -float('inf')

for i in range(5):

    model_name = f"NN_important_features_iteration_{i}"
    model = keras.Sequential()
    model.add(layers.Input(shape=(X_train_important.shape[1],)))
    model.add(layers.Dense(64, activation="tanh"))
    model.add(layers.Dense(32, activation="tanh"))
    model.add(layers.Dense(1))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.005),
        loss='mse',
        metrics=['mae'],
    )
    history, result = train_and_evaluate(model, model_name, X_train_important, y_train_important, X_val_important, y_val_important, X_test_important, y_test_important, epochs=300, batch_size=32, verbose=0)  

    test_r2 = result['R² Test']
    print(f"Iteration {i+1} - Test R²: {test_r2:.4f}")
    if test_r2 > best_test_score:
            best_test_score = test_r2
            best_model_important_features = model

best_model_important_features.save(f"best_model_important_features_iterations_{i+1}.h5")
print(f"\n bestes Modell gespeichert unter: best_model_important_features_iterations_{i+1}.h5")

Nochmal die besten vier Features aber diesmal speichern wir das Modell ;)

In [ ]:
important_features = ["MedInc", "AveOccup", "Latitude", "Longitude"] # aus rf_feauture_importance.png
# In Indizes umwandeln für Numpy Arrays
important_indices = [feature_names.index(feature) for feature in important_features]
X_train_important = X_train_standard[:, important_indices]
y_train_important = y_train_standard[:]
X_test_important = X_test_standard[:, important_indices]
y_test_important = y_test_standard[:]
X_val_important = X_val_standard[:, important_indices]
y_val_important = y_val_standard[:]

results = []
best_test_score = -float('inf')

for i in range(5):

    model_name = f"NN_4_important_features_iteration_{i}"
    model = keras.Sequential()
    model.add(layers.Input(shape=(X_train_important.shape[1],)))
    model.add(layers.Dense(64, activation="tanh"))
    model.add(layers.Dense(32, activation="tanh"))
    model.add(layers.Dense(1))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.005),
        loss='mse',
        metrics=['mae'],
    )
    history, result = train_and_evaluate(model, model_name, X_train_important, y_train_important, X_val_important, y_val_important, X_test_important, y_test_important, epochs=300, batch_size=32, verbose=0)  

    test_r2 = result['R² Test']
    print(f"Iteration {i+1} - Test R²: {test_r2:.4f}")
    if test_r2 > best_test_score:
            best_test_score = test_r2
            best_model_4_important_features = model

best_model_4_important_features.save(f"best_model_4_important_features_iterations_{i+1}.h5")
print(f"\n bestes Modell gespeichert unter: best_model_4_important_features_iterations_{i+1}.h5")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

model_important_features = keras.models.load_model("../results/nn_models/best_model_4_important_features_iterations_5.h5")
model_best_random_search = keras.models.load_model("../results/nn_models/best_model_random_search_iterations_100.h5")

y_pred_important_features = model_important_features.predict(X_test_important, verbose=0).flatten()
plot_predicted_vs_actual(
    y_test_important, 
    y_pred_important_features, 
    title="Bestes Modell mit wichtigsten vier Features - Residuenplot", 
    ax=axes[0]
)

y_pred_best_model = model_best_random_search.predict(X_test_standard, verbose=0).flatten()
plot_predicted_vs_actual(
    y_test_standard, 
    y_pred_best_model, 
    title="Bestes Modell mit allen Features - Residuenplot", 
    ax=axes[1]
)

fig.suptitle("Vergleich alle und vier wichtigsten Features", fontsize=14)
fig.tight_layout()
save_fig(fig, 'nn_4_vs_8_Features')
plt.show()

wichtigsten 5 Features testen -> eventuell noch mehr Informationen wie rauschen

In [ ]:
important_features = ["MedInc", "AveOccup", "Latitude", "Longitude", "HouseAge"] # aus rf_feauture_importance.png
# In Indizes umwandeln für Numpy Arrays
important_indices = [feature_names.index(feature) for feature in important_features]
X_train_important = X_train_standard[:, important_indices]
y_train_important = y_train_standard[:]
X_test_important = X_test_standard[:, important_indices]
y_test_important = y_test_standard[:]
X_val_important = X_val_standard[:, important_indices]
y_val_important = y_val_standard[:]

results = []
best_test_score = -float('inf')

for i in range(5):

    model_name = f"NN_5_important_features_iteration_{i}"
    model = keras.Sequential()
    model.add(layers.Input(shape=(X_train_important.shape[1],)))
    model.add(layers.Dense(64, activation="tanh"))
    model.add(layers.Dense(32, activation="tanh"))
    model.add(layers.Dense(1))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.005),
        loss='mse',
        metrics=['mae'],
    )
    history, result = train_and_evaluate(model, model_name, X_train_important, y_train_important, X_val_important, y_val_important, X_test_important, y_test_important, epochs=300, batch_size=32, verbose=0)  

    test_r2 = result['R² Test']
    print(f"Iteration {i+1} - Test R²: {test_r2:.4f}")
    if test_r2 > best_test_score:
            best_test_score = test_r2
            best_model_5_important_features = model

best_model_5_important_features.save(f"best_model_5_important_features_iterations_{i+1}.h5")
print(f"\n bestes Modell gespeichert unter: best_model_5_important_features_iterations_{i+1}.h5")

## Analyse des Modells in Hinblick auf Verbesserung der Hyperparameter

KI-Promt: Analyse weitermachen und Korrelationen und Outlier Analysen stärker verfiefen. Was kannst du mir als Variante empfehlen um eine outlier Analyse durchzuführen und Korrelationen zu entdecken?

### Residuen Histogramm

In [ ]:
important_features = ["MedInc", "AveOccup", "Latitude", "Longitude"] # aus rf_feauture_importance.png
# In Indizes umwandeln für Numpy Arrays
important_indices = [feature_names.index(feature) for feature in important_features]
X_test_important = X_test_standard[:, important_indices]

model_important_features = keras.models.load_model("../results/nn_models/best_model_4_important_features_iterations_5.h5")
model_best_random_search = keras.models.load_model("../results/nn_models/best_model_random_search_iterations_100.h5")

y_pred_important_features = model_important_features.predict(X_test_important, verbose=0).flatten()
y_pred_best_model = model_best_random_search.predict(X_test_standard, verbose=0).flatten()

plt.hist(y_test_important - y_pred_important_features, bins=50, alpha=0.6, color='g', label='Residuen-Histogramm für 4 Features')
plt.hist(y_test_standard - y_pred_best_model, bins=50, alpha=0.6, color='b', label='Residuen-Histogramm für alle Features')
plt.legend()
plt.xlabel('Residuen')
plt.ylabel('Häufigkeit')
plt.title('Vergleich der Residuenverteilung')
save_fig(plt, 'nn_residuals_histogramme_vergleich_featureanzahl')
plt.show()

Analyse:
- liegen symmetrisch um die Null: kein systematischer Fehler im Netz
- Außreißer gehen eher in die Positive Richtung, d.h. es gibt stärkere Fälle, bei denen der tatsächliche Preis höher als der vorhergesagte Preis ist
- über die Breite beider Verteilungen lässt sich dies schlecht händisch einschätzen. Im Bereich um die Null ist grün höher, was bei der selben Sample zahl eher für eine schmalere Verteilung spricht (kann aber auch durch die Form etc. nicht schmaler sein)

### Residuen vs. Feature Plot

Für die wichtigsten 4 Features

In [ ]:
residuals = y_pred_important_features - y_test_important
n_features = X_test_important.shape[1]
cols = 3
rows = int(np.ceil(n_features / cols))
fig, axes = plt.subplots(rows, cols, figsize=(18, 5 * rows))
axes = axes.flatten()

feature_zu_index = {feature: idx for idx, feature in enumerate(important_features)}

for i, feature in enumerate(important_features):
    idx = feature_zu_index[feature]
    ax = axes[i]
    ax.scatter(X_test_important[:, idx], residuals, alpha=0.5)
    ax.axhline(0, color='red', linestyle='--')
    ax.set_xlabel(feature)
    ax.set_ylabel('Residuen')
    ax.set_title(f'Residuen vs {feature}')

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

fig.tight_layout()
save_fig(fig, 'nn_residuals_vs_4 wichtigste_features')
plt.show

Interpretation des Ergebnisses mithile von (https://towardsdatascience.com/two-methods-for-performing-graphical-residuals-analysis-6899fd4c78e5/):

- keine U oder S-Form: Zusammenhänge korrekt gelernt
- keine Cluster
- kein Bias -> alle um Null herum